# Week 37: Character spans to BIO labels

In [1]:
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np

dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()

langlst = ["ar", "ko", "te"]
df_train_filtered = df_train[df_train["lang"].isin(langlst)]
df_validation_filtered = df_validation[df_validation["lang"].isin(langlst)]

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-multilingual-cased", use_fast=True
)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## Conversion functions

In [ ]:
def tokenize_with_offsets(text):
    encoded = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True,
        truncation=False,
    )
    tokens = encoded.tokens()
    offsets = encoded["offset_mapping"]
    return tokens, offsets


def character_span_to_bio(context, answer_start=None, answer_text=""):
    tokens, offsets = tokenize_with_offsets(context)
    labels = ["O"] * len(tokens)

    if answer_start is None or answer_text == "":
        return tokens, offsets, labels

    answer_end = answer_start + len(answer_text)
    if context[answer_start:answer_end] != answer_text:
        raise ValueError("The supplied answer text does not match the character span")

    covered = [
        index
        for index, (start, end) in enumerate(offsets)
        if start < answer_end and end > answer_start
    ]
    if not covered:
        raise ValueError("The answer does not overlap any token")
    # Unlike the lab, allow overlapping subwords when answer boundaries fall inside tokens.

    labels[covered[0]] = "B-ANS"
    for index in covered[1:]:
        labels[index] = "I-ANS"
    return tokens, offsets, labels


def bio_to_character_span(context, offsets, labels):
    if len(offsets) != len(labels):
        raise ValueError("Offsets and labels must have the same length")
    if not set(labels) <= {"O", "B-ANS", "I-ANS"}:
        raise ValueError("Only O, B-ANS and I-ANS labels are supported")

    answer_indices = [
        index for index, label in enumerate(labels) if label != "O"
    ]
    if not answer_indices:
        return None, ""

    start_index = answer_indices[0]
    expected_indices = list(range(start_index, start_index + len(answer_indices)))
    expected_labels = ["B-ANS"] + ["I-ANS"] * (len(answer_indices) - 1)
    if answer_indices != expected_indices:
        raise ValueError("The labels contain multiple or non-contiguous answer spans")
    if [labels[index] for index in answer_indices] != expected_labels:
        raise ValueError("The answer span must begin with B-ANS and continue with I-ANS")

    start = offsets[answer_indices[0]][0]
    end = offsets[answer_indices[-1]][1]
    return start, context[start:end]

## Worked example

In [3]:
context = "Ada Lovelace wrote the first algorithm."
answer_text = "Ada Lovelace"
answer_start = context.index(answer_text)

tokens, offsets, labels = character_span_to_bio(
    context, answer_start, answer_text
)
round_trip_start, round_trip_text = bio_to_character_span(
    context, offsets, labels
)

assert answer_start == 0
assert round_trip_start == answer_start
assert round_trip_text == answer_text
answer_labels = [label for label in labels if label != "O"]
assert len(answer_labels) > 1
assert answer_labels[0] == "B-ANS"
assert all(label == "I-ANS" for label in answer_labels[1:])

pd.DataFrame({"token": tokens, "offset": offsets, "label": labels})

,token,offset,label
0,Ada,"(0, 3)",B-ANS
1,Love,"(4, 8)",I-ANS
2,##lace,"(8, 12)",I-ANS
3,wrote,"(13, 18)",O
4,the,"(19, 22)",O
5,first,"(23, 28)",O
6,algorithm,"(29, 38)",O
7,.,"(38, 39)",O


## Check edge cases and prepare the dataset

In [4]:
context = "She visited Paris."
start = context.index("Paris")
tokens, offsets, labels = character_span_to_bio(context, start, "Paris")
assert bio_to_character_span(context, offsets, labels) == (start, "Paris")
assert tokens[-1] == "." and labels[-1] == "O"
tokens, offsets, labels = character_span_to_bio(context, None, "")
assert all(label == "O" for label in labels)
assert bio_to_character_span(context, offsets, labels) == (None, "")


def prepare_examples(df):
    examples = []
    for index, row in df.iterrows():
        start = int(row["answer_start"]) if row["answerable"] else None
        answer = row["answer"] if row["answerable"] else ""
        try:
            tokens, offsets, labels = character_span_to_bio(row["context"], start, answer)
        except ValueError as error:
            raise ValueError(f"BIO conversion failed at row {index}: {error}") from error
        # Record exact alignment instead of rejecting answers inside subwords.
        recovered = bio_to_character_span(row["context"], offsets, labels)
        aligned = recovered == (start, answer)
        question_tokens, _ = tokenize_with_offsets(row["question"])
        examples.append({
            "row_index": index,
            "aligned": aligned,
            "language": row["lang"],
            "question": row["question"],
            "context": row["context"],
            "question_tokens": question_tokens,
            "tokens": tokens,
            "offsets": offsets,
            "labels": labels,
            "answer_start": start,
            "answer": answer,
        })
    return examples


train_examples = prepare_examples(df_train_filtered)
validation_examples = prepare_examples(df_validation_filtered)
pd.DataFrame([
    {"split": "train", "converted": len(train_examples)},
    {"split": "validation", "converted": len(validation_examples)},
])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (546 > 512). Running this sequence through the model will result in indexing errors


,split,converted
0,train,6335
1,validation,1155


## Build token and question features 

In [ ]:
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import precision_recall_fscore_support


def token_features(tokens, index, question_tokens):
    word = tokens[index]
    features = {
        "bias": 1.0,
        "word.lower": word.lower(),
        "word.prefix2": word[:2].lower(),
        "word.suffix2": word[-2:].lower(),
        "word.suffix3": word[-3:].lower(),
        "word.istitle": word.istitle(),
        "word.isupper": word.isupper(),
        "word.isdigit": word.isdigit(),
        "contains_hyphen": "-" in word,
    }

    if index == 0:
        features["BOS"] = True
    else:
        previous = tokens[index - 1]
        features.update({
            "previous.lower": previous.lower(),
            "previous.istitle": previous.istitle(),
            "previous.isupper": previous.isupper(),
        })

    if index == len(tokens) - 1:
        features["EOS"] = True
    else:
        following = tokens[index + 1]
        features.update({
            "next.lower": following.lower(),
            "next.istitle": following.istitle(),
            "next.isupper": following.isupper(),
        })

    # Project addition: include the question for every context token.
    for token in question_tokens:
        features["question." + token.lower()] = True
    return features


def featurize_examples(examples):
    features, labels, lengths = [], [], []
    for example in examples:
        tokens = example["tokens"]
        lengths.append(len(tokens))
        for index in range(len(tokens)):
            features.append(token_features(tokens, index, example["question_tokens"]))
        labels.extend(example["labels"])
    return features, np.asarray(labels), lengths


vectorizer = DictVectorizer(sparse=True)
train_features, train_labels, train_lengths = featurize_examples(train_examples)
X_train = vectorizer.fit_transform(train_features)
validation_features, validation_labels, validation_lengths = featurize_examples(validation_examples)
X_validation = vectorizer.transform(validation_features)
X_train = X_train.astype(np.int32)
X_validation = X_validation.astype(np.int32)
print("Training matrix:", X_train.shape)
print("Validation matrix:", X_validation.shape)

## Constrained beam search

In [6]:
def valid_bio_transition(previous, current):
    if not current.startswith("I-"):
        return True
    if previous is None:
        return False
    entity_type = current[2:]
    return previous in {f"B-{entity_type}", f"I-{entity_type}"}


def constrained_beam_decode(token_log_probs, classes, beam_size=4):
    beam = [([], 0.0)]

    for scores in token_log_probs:
        candidates = []
        for sequence, sequence_score in beam:
            previous = sequence[-1] if sequence else None
            for column, label in enumerate(classes):
                # Project addition: allow at most one answer span.
                if label == "B-ANS" and "B-ANS" in sequence:
                    continue
                if valid_bio_transition(previous, label):
                    candidates.append((
                        sequence + [label],
                        sequence_score + float(scores[column]),
                    ))
        candidates.sort(key=lambda item: item[1], reverse=True)
        beam = candidates[:beam_size]

    return beam[0][0]


def split_by_lengths(values, lengths):
    sequences = []
    offset = 0
    for length in lengths:
        sequences.append(list(values[offset:offset + length]))
        offset += length
    assert offset == len(values)
    return sequences

## Compare answer-token F1 and exact span match with the empty-output baseline

In [ ]:
def evaluate_predictions(model_name, examples, predictions, seed=None):
    assert len(examples) == len(predictions)
    rows = []
    for language in ["all", "ar", "ko", "te"]:
        gold_tokens, predicted_tokens = [], []
        exact_matches = 0
        n = 0
        for example, labels in zip(examples, predictions):
            if language != "all" and example["language"] != language:
                continue
            assert len(labels) == len(example["labels"])
            gold_tokens.extend(label != "O" for label in example["labels"])
            predicted_tokens.extend(label != "O" for label in labels)
            predicted_span = bio_to_character_span(example["context"], example["offsets"], labels)
            exact_matches += predicted_span == (example["answer_start"], example["answer"])
            n += 1
        precision, recall, f1, _ = precision_recall_fscore_support(
            gold_tokens, predicted_tokens, average="binary", zero_division=0
        )
        rows.append({
            "model": model_name, "seed": seed, "language": language, "n": n,
            "token_precision": precision, "token_recall": recall, "token_f1": f1,
            "exact_span_match": exact_matches / n if n else np.nan,
        })
    return rows


empty_predictions = [["O"] * len(example["tokens"]) for example in validation_examples]
results = evaluate_predictions("Always empty", validation_examples, empty_predictions)
predictions_by_seed = {}
for seed in [42, 43]:
    classifier = SGDClassifier(
        loss="log_loss", alpha=1e-5, class_weight="balanced",
        max_iter=30, random_state=seed,
    )
    classifier.fit(X_train, train_labels)
    validation_log_probs = classifier.predict_log_proba(X_validation)
    log_prob_sequences = split_by_lengths(validation_log_probs, validation_lengths)
    predictions = []
    for scores in log_prob_sequences:
        predictions.append(constrained_beam_decode(scores, classifier.classes_, beam_size=4))
    predictions_by_seed[seed] = predictions
    results.extend(evaluate_predictions("Question-conditioned SGD", validation_examples, predictions, seed))

pd.DataFrame(results).round(4)